# CoMA ResNet CLIP-CAM Visualization

ViT 版 `visualize_CLIP_CAM.ipynb` 用 Transformer attention rollout 解释图文相似度。
ResNet-50 没有 patch attention，因此图像侧改为对 `layer4` 特征做 **Grad-CAM**（CLIP-CAM 的 CNN 对应实现），
文本侧仍使用 BERT self-attention rollout。

权重：
- Baseline: `/mnt/C/kxm/coma_ckpts/coma/baseline/2026_07_08_19_06_56/epoch=19-step=63139.ckpt`
- CoMA: `/mnt/C/kxm/coma_ckpts/coma/full_coma/2026_06_29_15_42_43/epoch=22-step=72633.ckpt`


In [ ]:
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
from matplotlib.backends.backend_agg import FigureCanvasAgg
import matplotlib.pyplot as plt
from pytorch_lightning import seed_everything
from transformers import BertTokenizer

try:
    from IPython.display import display, HTML
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

try:
    from captum.attr import visualization
except ImportError:
    visualization = None


def find_repo_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "coma" / "models" / "coma" / "module.py").exists():
            return p
    return Path("/mnt/E/code/CoMA")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from coma.datasets.utils import get_imgs
from coma.models.coma.module import CoMA
from coma.models.backbones.med import BertSelfAttention

seed_everything(42)

In [ ]:
BASELINE_CKPT = (
    "/mnt/C/kxm/coma_ckpts/coma/baseline/2026_07_08_19_06_56/"
    "epoch=19-step=63139.ckpt"
)
COMA_CKPT = (
    "/mnt/C/kxm/coma_ckpts/coma/full_coma/2026_06_29_15_42_43/"
    "epoch=22-step=72633.ckpt"
)
BERT_DIR = REPO_ROOT / "model_files" / "Bio_ClinicalBERT"
MIMIC_JPG_DIR = Path(
    "/mnt/E/dataset/MIMIC-CXR-JPG/physionet.org/files/mimic-cxr-jpg/2.0.0/files"
)

MAX_LENGTH = 112
CROP_SIZE = 224
TEMPERATURE = 0.07
START_LAYER_TEXT = -1
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = BertTokenizer.from_pretrained(str(BERT_DIR))
baseline_model = CoMA.load_from_checkpoint(BASELINE_CKPT, strict=False).to(device).eval()
CoMA_model = CoMA.load_from_checkpoint(COMA_CKPT, strict=False).to(device).eval()

# BERT config already returns attentions; also keep a module-level copy for rollout.
if not getattr(BertSelfAttention, "_coma_resnet_cam_patched", False):
    _orig_bert_self_forward = BertSelfAttention.forward

    def _bert_self_forward_save(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        past_key_value=None,
        output_attentions=False,
    ):
        outputs = _orig_bert_self_forward(
            self,
            hidden_states,
            attention_mask,
            head_mask,
            encoder_hidden_states,
            encoder_attention_mask,
            past_key_value,
            True,
        )
        if getattr(self, "save_attention", False) and isinstance(outputs, tuple) and len(outputs) > 1:
            self.save_attention_map(outputs[1])
        return outputs

    BertSelfAttention.forward = _bert_self_forward_save
    BertSelfAttention._coma_resnet_cam_patched = True


def enable_text_attention_save(model):
    for layer in model.text_encoder_q.model.encoder.layer:
        layer.attention.self.save_attention = True


enable_text_attention_save(baseline_model)
enable_text_attention_save(CoMA_model)
print("device:", device)
print("baseline img_encoder:", baseline_model.hparams.img_encoder)
print("CoMA img_encoder:", CoMA_model.hparams.img_encoder)

In [ ]:
class visDataTransforms(object):
    def __init__(self, crop_size: int = 224):
        self.data_transforms = transforms.Compose([
            transforms.CenterCrop(crop_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

    def __call__(self, image):
        return self.data_transforms(image)


def get_images(imgs_path):
    imgs = []
    for img_path in imgs_path:
        imgs.append(get_imgs(img_path, 256, visDataTransforms(CROP_SIZE)).to(device))
    stacked = torch.stack(imgs)
    stacked.requires_grad_(True)
    return stacked


def get_texts(fplist):
    texts = {"patient": [], "study": [], "texts": []}
    for fpath in fplist:
        fpath = str(fpath)
        patient = fpath.split("/")[-2]
        name, _ = os.path.splitext(os.path.basename(fpath))
        name = name.split("_")[0]
        with open(fpath, "r") as f:
            ori_text = f.readlines()

        has_findings = any("FINDINGS" in line for line in ori_text)
        if has_findings:
            text = None
            for line in ori_text:
                if "FINDINGS" in line:
                    text = line.split("FINDINGS", 1)[-1].lstrip(":").strip()
                    continue
                if "IMPRESSION" in line:
                    break
                if text is None or line.isspace():
                    continue
                text += " " + line.replace("\n", "")
        else:
            # CCD reports are one clause per line, without FINDINGS headers.
            text = " ".join(line.strip() for line in ori_text if line.strip())

        texts["patient"].append(patient)
        texts["study"].append(name)
        texts["texts"].append((text or "").strip())
    return texts

In [ ]:
class Layer4Activations:
    def __init__(self):
        self.features = None

    def __call__(self, module, inputs, output):
        self.features = output


def calculate(model, images, texts):
    img_feat_q, _, _ = model._encode_images(images)
    img_emb_q = F.normalize(model.img_encoder_q.global_embed(img_feat_q), dim=-1)
    token_type_ids = texts.get("token_type_ids", torch.zeros_like(texts["input_ids"]))
    report_feat_q, _, _, _ = model.text_encoder_q(
        texts["input_ids"], texts["attention_mask"], token_type_ids)
    report_emb_q = F.normalize(model.text_encoder_q.global_embed(report_feat_q), dim=-1)
    logits_per_image = img_emb_q.mm(report_emb_q.t()) / TEMPERATURE
    return logits_per_image


def gradcam_from_layer4(score, activations):
    grads = torch.autograd.grad(score, activations, retain_graph=True)[0]
    weights = grads.mean(dim=(2, 3), keepdim=True)
    cam = (weights * activations).sum(dim=1).clamp(min=0)
    return cam


def text_attention_rollout(score, model, batch_size, device, start_layer_text=START_LAYER_TEXT):
    text_attn_blocks = model.text_encoder_q.model.encoder.layer
    if start_layer_text == -1:
        start_layer_text = len(text_attn_blocks) - 1

    attn0 = text_attn_blocks[start_layer_text].attention.self.get_attention_map()
    num_tokens = attn0.shape[-1]
    R_text = torch.eye(num_tokens, num_tokens, dtype=attn0.dtype, device=device)
    R_text = R_text.unsqueeze(0).expand(batch_size, num_tokens, num_tokens)

    for i, blk in enumerate(text_attn_blocks):
        if i < start_layer_text:
            continue
        cam = blk.attention.self.get_attention_map()
        grad = torch.autograd.grad(score, [cam], retain_graph=True)[0].detach()
        cam = cam.detach()
        cam = cam.reshape(-1, cam.shape[-1], cam.shape[-1])
        grad = grad.reshape(-1, grad.shape[-1], grad.shape[-1])
        cam = grad * cam
        cam = cam.reshape(batch_size, -1, cam.shape[-1], cam.shape[-1])
        cam = cam.clamp(min=0).mean(dim=1)
        R_text = R_text + torch.bmm(cam, R_text)
    return R_text


def interpret(image, texts, model, device, start_layer_text=START_LAYER_TEXT):
    model.zero_grad()
    act_store = Layer4Activations()
    handle = model.img_encoder_q.model.layer4.register_forward_hook(act_store)
    try:
        with torch.enable_grad():
            logits_per_image = calculate(model, image, texts)
            batch_size = texts["input_ids"].shape[0]
            score = logits_per_image.diag().sum()
            image_relevance = gradcam_from_layer4(score, act_store.features)
            text_relevance = text_attention_rollout(
                score, model, batch_size, device, start_layer_text)
    finally:
        handle.remove()
    return text_relevance, image_relevance


def show_heatmap_on_text(text, text_encoding, R_text, sentence_split=True):
    cls_idx = torch.where(text_encoding == 101)[0]
    sep_idx = torch.where(text_encoding == 102)[0]
    CLS_idx = cls_idx[0] if cls_idx.numel() else torch.tensor(0, device=text_encoding.device)
    SEP_idx = sep_idx[0] if sep_idx.numel() else torch.tensor(len(text_encoding) - 1, device=text_encoding.device)
    R_text = R_text[CLS_idx, CLS_idx + 1:SEP_idx]
    if R_text.numel() == 0 or R_text.sum() == 0:
        R_text = torch.ones(max(int(SEP_idx - CLS_idx - 1), 1), device=R_text.device)
    text_scores = R_text / R_text.sum()
    text_scores = text_scores.flatten()
    text_tokens = tokenizer(text)["input_ids"][1:-1]
    text_tokens_decoded = [tokenizer.decode([a]) for a in text_tokens]
    text_tokens_decoded = text_tokens_decoded[:MAX_LENGTH - 2]
    n = min(len(text_tokens_decoded), text_scores.numel())
    text_scores = text_scores[:n]
    text_tokens_decoded = text_tokens_decoded[:n]
    if any(["##" in t for t in text_tokens_decoded]):
        n_text_scores = []
        n_text_tokens_decoded = []
        for i, token in enumerate(text_tokens_decoded):
            if "##" in token and i != 0:
                n_text_scores[-1] += text_scores.tolist()[i]
                n_text_tokens_decoded[-1] += text_tokens_decoded[i].replace("##", "")
            else:
                n_text_scores.append(text_scores.tolist()[i])
                n_text_tokens_decoded.append(text_tokens_decoded[i])
        text_scores = torch.tensor(n_text_scores).to(text_scores)
        text_tokens_decoded = n_text_tokens_decoded
    if sentence_split:
        n_text_scores = []
        n_text_tokens_decoded = []
        for i, token in enumerate(text_tokens_decoded):
            if i == 0:
                n_text_scores.append(text_scores.tolist()[i])
                n_text_tokens_decoded.append(text_tokens_decoded[i])
            else:
                n_text_scores[-1] += text_scores.tolist()[i]
                if token not in [",", "."]:
                    n_text_tokens_decoded[-1] += " " + text_tokens_decoded[i]
                else:
                    n_text_tokens_decoded[-1] += text_tokens_decoded[i]
                    n_text_scores.append(0)
                    n_text_tokens_decoded.append("")
        text_scores = torch.tensor(n_text_scores).to(text_scores)
        text_tokens_decoded = n_text_tokens_decoded
    return [(s, [1 - c, 1, 1 - c]) for s, c in zip(text_tokens_decoded, text_scores.cpu().numpy().tolist())]

In [ ]:
def get_cam_images(image_relevances, image, vis_size=224):
    def show_cam_on_image(img, mask):
        heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
        heatmap = np.float32(heatmap) / 255
        cam = heatmap + np.float32(img)
        cam = cam / np.max(cam)
        return cam

    image = image.permute(1, 2, 0).data.cpu().numpy()
    image = (image - image.min()) / (image.max() - image.min() + 1e-8)
    cam_images = []
    for image_relevance in image_relevances:
        rel = image_relevance.detach()
        if rel.ndim == 1:
            dim = int(rel.numel() ** 0.5)
            rel = rel.reshape(1, 1, dim, dim)
        elif rel.ndim == 2:
            rel = rel.unsqueeze(0).unsqueeze(0)
        else:
            rel = rel.reshape(1, 1, rel.shape[-2], rel.shape[-1])
        rel = F.interpolate(rel.float(), size=vis_size, mode="bilinear", align_corners=False)
        rel = rel.reshape(vis_size, vis_size).cpu().numpy()
        rel = (rel - rel.min()) / (rel.max() - rel.min() + 1e-8)
        vis = show_cam_on_image(image, rel)
        vis = np.uint8(255 * vis)
        vis = cv2.cvtColor(np.array(vis), cv2.COLOR_RGB2BGR)
        cam_images.append(vis)
    return cam_images


def add_wrapped_color_text(ax, text_segments, start_pos, max_line_width, fontsize=12):
    canvas = FigureCanvasAgg(ax.figure)
    renderer = canvas.get_renderer()

    x, y = start_pos
    line_height = fontsize * 0.004
    current_line_width = 0

    for text, color in text_segments:
        words = text.split()
        for word in words:
            temp_text = ax.text(
                x, y, word + " ",
                ha="left", va="center",
                transform=ax.transAxes,
                fontsize=fontsize,
                bbox={
                    "facecolor": "white",
                    "edgecolor": "none",
                    "pad": 0,
                    "alpha": 0,
                    "boxstyle": "Round,pad=0",
                },
            )
            bb = temp_text.get_window_extent(renderer=renderer)
            bb_ax = bb.transformed(ax.transAxes.inverted())
            word_width = bb_ax.x1 - bb_ax.x0
            temp_text.remove()

            if current_line_width + word_width > max_line_width:
                y -= line_height
                x = start_pos[0]
                current_line_width = 0

            ax.text(
                x, y, word + " ",
                ha="left", va="center",
                transform=ax.transAxes,
                fontsize=fontsize,
                bbox={
                    "facecolor": color,
                    "edgecolor": "none",
                    "pad": 0,
                    "boxstyle": "Round,pad=0",
                },
            )
            x += word_width
            current_line_width += word_width
    ax.axis("off")


def print_text_visual_bar(vis_data_records):
    if not HAS_IPYTHON or visualization is None:
        return None
    dom = ["<table width: 100%>"]
    rows = ["<th style='text-align: center'>Word Importance</th>"]
    for datarecord in vis_data_records:
        rows.append("".join([
            "<tr>",
            visualization.format_word_importances(
                datarecord.raw_input_ids, datarecord.word_attributions),
            "<tr>",
        ]))
    dom.append(
        '<div style="border-top: 1px solid; margin-top: 5px; '
        'padding-top: 5px; display: inline-block">'
    )
    dom.append("<b>Legend: </b>")
    for value, label in zip([-1, 0, 1], ["Negative", "Neutral", "Positive"]):
        dom.append(
            '<span style="display: inline-block; width: 10px; height: 10px; '
            'border: 1px solid; background-color: '
            '{value}"></span> {label}  '.format(
                value=visualization._get_color(value), label=label
            )
        )
    dom.append("</div>")
    dom.append("".join(rows))
    dom.append("</table>")
    html = HTML("".join(dom))
    display(html)
    return html

In [ ]:
patients = [
    "p10/p10002428/s50027225/e8c44648-ff02beea-3d5ff638-dec79b01-7df71a69.jpg",
    "p10/p10012261/s56872341/edeabd5b-139705f9-7fdef9a3-c6bdb0ed-000cef8f.jpg",
    "p10/p10013653/s53212555/3b9f9742-adbf9136-b41b8572-ed58df70-c59d06da.jpg",
]

imgs_path = [str(MIMIC_JPG_DIR / p) for p in patients]
reports_path = [
    img_path.replace("/" + os.path.basename(img_path), "_ccd.txt")
    for img_path in imgs_path
]

imgs = get_images(imgs_path)
texts = get_texts(fplist=reports_path)["texts"]
print("reports:", texts)

text = tokenizer(
    texts,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
)
text = {k: v.to(device) for k, v in text.items()}

_, b_R_image = interpret(model=baseline_model, image=imgs, texts=text, device=device)
c_R_text, c_R_image = interpret(model=CoMA_model, image=imgs, texts=text, device=device)
textid = text["input_ids"]
batch_size = textid.shape[0]

fig, axs = plt.subplots(batch_size, 4, figsize=(24, 16))
if batch_size == 1:
    axs = np.expand_dims(axs, 0)

for i in range(batch_size):
    text_segments = show_heatmap_on_text(texts[i], textid[i], c_R_text[i])
    cam_images = get_cam_images([b_R_image[i], c_R_image[i]], imgs[i])

    axs[i][0].imshow(Image.open(imgs_path[i]).convert("RGB"))
    axs[i][0].axis("off")

    for j, cam_img in enumerate(cam_images):
        axs[i][j + 1].imshow(cam_img)
        axs[i][j + 1].axis("off")

    add_wrapped_color_text(
        ax=axs[i][3],
        text_segments=text_segments,
        start_pos=(0.05, 0.95),
        max_line_width=0.95,
        fontsize=16,
    )

axs[0][0].set_title("original image", fontsize=20)
axs[0][1].set_title("baseline", fontsize=20)
axs[0][2].set_title("CoMA", fontsize=20)
axs[0][3].set_title("Image report", fontsize=20)

out_path = REPO_ROOT / "coma" / "visualization" / "vis_resnet_clip_cam.png"
plt.savefig(out_path, bbox_inches="tight", dpi=150)
plt.show()
print("saved:", out_path)